# Scraping Song Lyrics

In this lab, you will scrape the lyrics of songs by your favorite artist.

In Part B of this lab, you will train a model on these lyrics so that you can generate a new "song" in the style of that artist!

Find a web site or REST API that contains song lyrics by your favorite artist. If you decide to scrape a webpage, be sure to document any ethical decisions you made, such as:
- not making more requests to the website than necessary
- staggering your requests **(If you do not stagger your requests with a time delay, you will earn an automatic 0 for the assignment.)**
- respecting the robots.txt file.

Some words of wisdom from our experience:

- The [genius.com API](https://docs.genius.com/) does not return the lyrics, only a link to a webpage with the lyrics.
- The free version of the [MusixMatch API](https://developer.musixmatch.com/plans) only returns the first 30% of the lyrics.
- If you are web scraping, find a web page that has links to all the songs by the artist, like [this page for the Steve Miller Band](https://www.azlyrics.com/s/stevemillerband.html). [_Note:_ It appears that `azlyrics.com` blocks web scraping, so you may have to find a different lyrics web site.] Then, you can scrape this page, extract the hyperlinks, and issue new HTTP requests to each hyperlink to get each song.
- Use inspect element/browser web tools to find where the lyrics are stored
- Use beautifulsoup for data cleaning

Create a `DataFrame`, where each row represents a song by that artist. The `DataFrame` should have two columns:
- The first should contain the title of the song.
- The second should contain the complete lyrics of the song as a string.
    - It should **not** contain any extraneous text or HTML tags. (Such as verse markings or HTML formatted line breaks)

Display the `DataFrame` in the Colab. Save the `DataFrame` as a CSV file, and download the CSV. (You will need it in Part B.)

In [1]:
# import sys
# !{sys.executable} -m pip install lyricsgenius
import lyricsgenius
import pandas as pd
import time

token = "REDACTED"

In [2]:
# connect to genius
genius = lyricsgenius.Genius(
    token,
    timeout=15,
    retries=3,
    remove_section_headers=True,
    skip_non_songs=True,
    excluded_terms=["(Remix)", "(Live)", "(Demo)", "(Instrumental)"]
)

In [3]:
# search michael jackson
artist = genius.search_artist(
    "Michael Jackson",
    max_songs=50,
    sort="popularity"
)

In [4]:
# build lists
titles = []
lyrics_list = []

for song in artist.songs:
    titles.append(song.title)
    lyrics_list.append(song.lyrics)

    time.sleep(0.5)

# create the DataFrame
songs_df = pd.DataFrame({
    "title": titles,
    "lyrics": lyrics_list
})

# replace \n
songs_df["lyrics"] = songs_df["lyrics"].str.replace(r"\\n", "\n", regex=True)

songs_df.head(10)

,title,lyrics
0,Billie Jean,She was more like a beauty queen from a movie ...
1,Smooth Criminal,"Ow\nCha\nShoo-ca, choo-ca\n\nAs he came into t..."
2,Wanna Be Startin’ Somethin’,Ooh\n\nI said you wanna be startin' somethin'\...
3,Heal the World,"Think about, um, the generations, and, uh, say..."
4,Thriller,It's close to midnight\nAnd something evil's l...
5,They Don’t Care About Us,All I wanna say is that they don't really care...
6,Human Nature,Why?\n\nLooking out across the nighttime\nThe ...
7,Man in the Mirror,I'm gonna make a change for once in my life\nI...
8,Beat It,"They told him, ""Don't you ever come around her..."
9,Chicago,"Ah, I met her on my way to Chicago\nWhere she ..."


In [5]:
# save CSV
songs_df.to_csv(
    "michael_jackson_lyrics.csv",
    index=False,
    encoding="utf-8-sig" # needed so apostrophes load properly
)

## Ethical Decisions

I used the Genius API through the lyricsgenius Python package rather than manually scraping HTML pages. This reduced unnecessary requests to the website and allowed lyrics to be collected in a more structured way.

I limited the dataset to 50 songs and included a 0.5s delay between requests using `time.sleep()` to avoid excessive traffic and unnecessary repeated requests.